In [0]:
%run ./utils/cleaning_transformation_to_silver_utils

In [0]:
%run ./utils/write_to_delta_utils

In [0]:
%restart_python or dbutils.library.restartPython()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.sql("CREATE SCHEMA IF NOT EXISTS novamart.silver")

delta_write_options = {
    "tblproperties.delta.autoOptimize.optimizeWrite": "true",
    "tblproperties.delta.autoOptimize.autoCompact": "true",
    "mergeSchema": "true",
    "tblproperties.delta.enableChangeDataFeed": "true"
}

# ----------------------------------------------------
# 1. READ BRONZE & TRANSFORM (Passing to Functions)
# ----------------------------------------------------
df_silver_sql_customers = bronze_sql_customers_silver(spark.read.table("novamart.bronze.sql_customers"))
df_silver_crm_customers = bronze_crm_customers_silver(spark.read.table("novamart.bronze.crm_customers"))
df_silver_sql_products = bronze_sql_products_silver(spark.read.table("novamart.bronze.sql_products"))
df_silver_sale_transactions = bronze_sql_sale_transactions(spark.read.table(
                                                "novamart.bronze.bronze_sql_sale_transactions")
                                                 )
df_silver_clickstream = bronze_clickstream(spark.read.table("novamart.bronze.clickstream"))


# ----------------------------------------------------
# 2. WRITE DIMENSION TABLES (Using MERGE/UPSERT)
# ----------------------------------------------------


# ----------------------------------------------------
# 2. WRITE sql customers table
# ----------------------------------------------------

write_delta_table(
    df=df_silver_sql_customers,
    table_name= "novamart.silver.sql_customers",
    write_mode="merge",
    merge_key= "customer_id",
    cluster_keys= ["customer_id"]
)


# ----------------------------------------------------
# 2. WRITE crm customers table
# ----------------------------------------------------

write_delta_table(
    df=df_silver_crm_customers,
    table_name= "novamart.silver.crm_customers",
    write_mode="merge",
    merge_key= "customer_id",
    cluster_keys= ["customer_id"]
)

# ----------------------------------------------------
# 2. WRITE sql products table
# ----------------------------------------------------

write_delta_table(
    df=df_silver_sql_products,
    table_name= "novamart.silver.sql_products",
    write_mode="merge",
    merge_key= "product_id",
    cluster_keys= ["product_id"]
)

# ----------------------------------------------------
# 2. WRITE DIMENSION TABLES (Using APPEND/OVERWRITE)
# ----------------------------------------------------

# ----------------------------------------------------
# 2. WRITE sql sale transactions table
# ----------------------------------------------------

write_delta_table(
    df=df_silver_sale_transactions,
    table_name= "novamart.silver.sql_sale_transactions",
    write_mode="append",
    cluster_keys= ["customer_id", "product_id"]
)
    

# ----------------------------------------------------
# 2. WRITE clickstream table
# ----------------------------------------------------

write_delta_table(
    df=df_silver_clickstream,
    table_name= "novamart.silver.clickstream",
    write_mode="append",
    cluster_keys= ["customer_id", "product_id"]
)